# IVF Two-Level Index — Performance Evaluation

Evaluates the `TwoLevelIVFIndex` at 1M and 10M records.

**Sections**
1. Config & imports
2. Deterministic vector generator (idempotent resume)
3. Idempotent insert runner
4. Insert 1M records
5. Insert 10M records *(slow — run separately)*
6. Storage metrics
7. Clustering quality (WSS, size distribution, inter-centroid spread)
8. Search quality (Recall@k, latency)
9. Summary

In [ ]:
import sys, os, json, time, math
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Add project root so Storage package resolves
PROJECT_ROOT = Path("../..")
sys.path.insert(0, str(PROJECT_ROOT.resolve()))

from Storage.IVF.two_level_index import TwoLevelIVFIndex

try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

print("Imports OK")

In [ ]:

DIM        = 384
BATCH_SIZE = 10_000   # vectors generated per deterministic batch

TARGETS = {
    "1M":  1_000_000,
    "10M": 10_000_000,
}

# Each scale gets its own isolated directory — safe to run in parallel or restart
DATA_ROOT = Path("./eval_data")
INDEX_ROOTS = {scale: DATA_ROOT / f"ivf_{scale}" for scale in TARGETS}

# IVF hyperparameters (keep identical across scales for a fair comparison)
IVF_KWARGS = dict(
    dimension              = DIM,
    max_level1_clusters    = 500,
    max_level2_per_level1  = 20,
    level1_probes          = 12,
    level2_probes          = 6,
    new_level1_threshold   = 0.62,
    new_level2_threshold   = 0.78,
)

# Evaluation parameters
N_QUERY_VECTORS  = 200   # for Recall@k
TOP_K            = 10    # k for Recall@k
WSS_SAMPLE_CLUSTERS = 500  # max level2 clusters to use for WSS (set None for all)
SILHOUETTE_SAMPLES  = 2_000

for p in INDEX_ROOTS.values():
    p.mkdir(parents=True, exist_ok=True)

print("Data dirs:", {k: str(v) for k, v in INDEX_ROOTS.items()})

## Deterministic Vector Generator

Each batch of `BATCH_SIZE` vectors is seeded by its **batch index**, so:
- The same vector at index `i` is always reproduced as `batch[i // BATCH_SIZE][i % BATCH_SIZE]`.
- After a crash we only need `vector_count` from the manifest to know exactly where to resume.

In [ ]:
def make_batch(batch_idx: int, size: int = BATCH_SIZE, dim: int = DIM) -> np.ndarray:
    """Return `size` L2-normalised float32 vectors, deterministic for `batch_idx`."""
    rng  = np.random.default_rng(seed=batch_idx)
    vecs = rng.standard_normal((size, dim)).astype(np.float32)
    norms = np.linalg.norm(vecs, axis=1, keepdims=True)
    return vecs / norms


def make_query_vectors(n: int = N_QUERY_VECTORS, dim: int = DIM, seed: int = 999_999_999) -> np.ndarray:
    """Fresh query vectors that were never inserted (use a seed outside the insert range)."""
    rng  = np.random.default_rng(seed=seed)
    vecs = rng.standard_normal((n, dim)).astype(np.float32)
    norms = np.linalg.norm(vecs, axis=1, keepdims=True)
    return vecs / norms


# Quick sanity check
_b0 = make_batch(0)
_b0b = make_batch(0)   # must be identical
assert np.allclose(_b0, _b0b), "Generator not deterministic!"
print(f"Batch shape: {_b0.shape}  |  norms ≈ 1: {np.allclose(np.linalg.norm(_b0, axis=1), 1.0)}")

## Idempotent Insert

Reads `vector_count` from the existing manifest, computes the first un-inserted batch index and the offset within that batch, then continues inserting until the target is reached.

In [ ]:
def insert_to_target(
    index: TwoLevelIVFIndex,
    target: int,
    batch_size: int = BATCH_SIZE,
    report_every: int = 100_000,
) -> None:
    already = index.vector_count
    if already >= target:
        print(f"Already at {already:,} / {target:,} — nothing to do.")
        return

    print(f"Resuming from {already:,} → {target:,}  ({target - already:,} to insert)")

    first_batch = already // batch_size
    first_offset = already % batch_size
    remaining   = target - already

    t0 = time.perf_counter()
    inserted = 0
    next_report = report_every

    with tqdm(total=remaining, unit="vec", unit_scale=True, desc="Inserting") as pbar:
        batch_idx = first_batch
        while inserted < remaining:
            batch = make_batch(batch_idx)

            # First batch: skip vectors already inserted
            if batch_idx == first_batch and first_offset > 0:
                batch = batch[first_offset:]

            # Last (partial) batch: trim to not exceed target
            left = remaining - inserted
            if len(batch) > left:
                batch = batch[:left]

            for vec in batch:
                index.insert(vec)
                inserted += 1
                pbar.update(1)

                if inserted >= next_report:
                    elapsed = time.perf_counter() - t0
                    rate    = inserted / elapsed
                    eta_s   = (remaining - inserted) / rate if rate > 0 else float("inf")
                    pbar.set_postfix(
                        rate=f"{rate:,.0f}/s",
                        eta=f"{eta_s/60:.1f}min",
                    )
                    next_report += report_every

            batch_idx += 1

    elapsed = time.perf_counter() - t0
    print(f"\nDone. Total: {index.vector_count:,}  |  Time: {elapsed/60:.1f} min  |  Rate: {inserted/elapsed:,.0f} vec/s")

## Insert 1M Records

Safe to re-run — will resume from the current `vector_count`.

In [ ]:
idx_1M = TwoLevelIVFIndex(root_dir=str(INDEX_ROOTS["1M"]), **IVF_KWARGS)
print(f"Current 1M index state: {idx_1M.vector_count:,} vectors")
insert_to_target(idx_1M, target=TARGETS["1M"])

## Insert 10M Records

**Warning**: ~15 GB of vector data on disk; insert time is ~10× the 1M run.  
Safe to stop and resume at any point.

In [ ]:
idx_10M = TwoLevelIVFIndex(root_dir=str(INDEX_ROOTS["10M"]), **IVF_KWARGS)
print(f"Current 10M index state: {idx_10M.vector_count:,} vectors")
insert_to_target(idx_10M, target=TARGETS["10M"])

---
## Storage Metrics

In [ ]:
def dir_size_bytes(path: Path) -> int:
    return sum(f.stat().st_size for f in path.rglob("*") if f.is_file())


def storage_report(scale: str) -> dict:
    root = INDEX_ROOTS[scale]
    manifest = json.loads((root / "manifest.json").read_text())
    n = manifest["vector_count"]
    dim = manifest["dimension"]

    files = {
        "vectors.dat":            root / "vectors.dat",
        "level1_centroids.dat":   root / "level1_centroids.dat",
        "level2_centroids.dat":   root / "level2_centroids.dat",
        "layout.json":            root / "layout.json",
        "manifest.json":          root / "manifest.json",
    }
    postings_dir = root / "postings"

    sizes = {name: p.stat().st_size for name, p in files.items() if p.exists()}
    sizes["postings/"] = dir_size_bytes(postings_dir) if postings_dir.exists() else 0
    sizes["TOTAL"]     = sum(sizes.values())

    layout = json.loads((root / "layout.json").read_text())
    n_l1 = len(layout["level1_to_level2"])
    n_l2 = sum(len(v) for v in layout["level1_to_level2"])
    n_posting_files = len(list(postings_dir.glob("*.dat"))) if postings_dir.exists() else 0

    return {
        "scale":           scale,
        "vectors":         n,
        "dim":             dim,
        "level1_clusters": n_l1,
        "level2_clusters": n_l2,
        "posting_files":   n_posting_files,
        "sizes_bytes":     sizes,
    }


def fmt_bytes(b: int) -> str:
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if b < 1024:
            return f"{b:.1f} {unit}"
        b /= 1024
    return f"{b:.1f} PB"


reports = {}
for scale in TARGETS:
    root = INDEX_ROOTS[scale]
    if not (root / "manifest.json").exists():
        print(f"[{scale}] index not built yet — skip")
        continue
    r = storage_report(scale)
    reports[scale] = r
    print(f"\n{'─'*50}")
    print(f"  Scale : {r['scale']}  ({r['vectors']:,} vectors, dim={r['dim']})")
    print(f"  L1 clusters : {r['level1_clusters']}")
    print(f"  L2 clusters : {r['level2_clusters']}  ({r['posting_files']} posting files)")
    print(f"  Storage breakdown:")
    for name, sz in r["sizes_bytes"].items():
        print(f"    {name:<28s} {fmt_bytes(sz):>10}")

In [ ]:
# Storage breakdown bar chart
if reports:
    fig, axes = plt.subplots(1, len(reports), figsize=(7 * len(reports), 5))
    if len(reports) == 1:
        axes = [axes]

    for ax, (scale, r) in zip(axes, reports.items()):
        sizes = {k: v / (1024**3) for k, v in r["sizes_bytes"].items() if k != "TOTAL"}
        ax.barh(list(sizes.keys()), list(sizes.values()), color="steelblue")
        ax.set_xlabel("Size (GB)")
        ax.set_title(f"Storage breakdown — {scale}  ({fmt_bytes(r['sizes_bytes']['TOTAL'])})")
        ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))

    plt.tight_layout()
    plt.savefig(DATA_ROOT / "storage_breakdown.png", dpi=120)
    plt.show()

---
## Clustering Quality Metrics

| Metric | What it measures |
|--------|------------------|
| **WSS** (Within-cluster sum of cosine distances) | Compactness — lower is better |
| **Size CV** (coeff. of variation of cluster sizes) | Balance — lower is better |
| **Empty clusters** | Wasted partitions |
| **Inter-centroid spread** | Separation of L1 centroids — higher is better |
| **Silhouette score (sample)** | Combined cohesion + separation ∈ [−1,1] — higher is better |

In [ ]:
def load_centroids(path: Path, dim: int) -> np.ndarray:
    if not path.exists() or path.stat().st_size == 0:
        return np.empty((0, dim), dtype=np.float32)
    raw = np.fromfile(str(path), dtype=np.float32)
    return raw.reshape(-1, dim)


def load_posting(postings_dir: Path, cluster_id: int) -> np.ndarray:
    p = postings_dir / f"c_{cluster_id}.dat"
    if not p.exists() or p.stat().st_size == 0:
        return np.empty(0, dtype=np.uint32)
    return np.fromfile(str(p), dtype=np.uint32)


def clustering_metrics(scale: str, wss_sample: int | None = WSS_SAMPLE_CLUSTERS) -> dict:
    root         = INDEX_ROOTS[scale]
    manifest     = json.loads((root / "manifest.json").read_text())
    layout       = json.loads((root / "layout.json").read_text())
    n            = manifest["vector_count"]
    dim          = manifest["dimension"]
    postings_dir = root / "postings"

    level1 = load_centroids(root / "level1_centroids.dat", dim)
    level2 = load_centroids(root / "level2_centroids.dat", dim)

    l2_sizes  = np.array(layout["level2_counts"], dtype=np.int64)  # actual insert counts
    n_l2      = len(l2_sizes)
    empty_l2  = int(np.sum(l2_sizes == 0))

    # ── Cluster size statistics ───────────────────────────────────────────────
    nonempty = l2_sizes[l2_sizes > 0]
    size_mean = float(nonempty.mean())
    size_std  = float(nonempty.std())
    size_cv   = size_std / size_mean if size_mean > 0 else float("nan")
    size_max  = int(nonempty.max())
    size_min  = int(nonempty.min())

    # ── Inter-centroid spread (L1) ─────────────────────────────────────────────
    # Average pairwise cosine distance between L1 centroids (sample 200)
    n_sample = min(200, level1.shape[0])
    rng_spread = np.random.default_rng(0)
    idx_sample = rng_spread.choice(level1.shape[0], size=n_sample, replace=False)
    l1_sample  = level1[idx_sample]   # (n_sample, dim)  — already normalised
    cos_sim_mat = l1_sample @ l1_sample.T   # (n_sample, n_sample)
    # exclude diagonal (self-similarity = 1)
    mask = ~np.eye(n_sample, dtype=bool)
    inter_cos_dist = float(1.0 - cos_sim_mat[mask].mean())


    # Sample at most `wss_sample` non-empty L2 clusters
    nonempty_ids = np.where(l2_sizes > 0)[0]
    if wss_sample is not None and len(nonempty_ids) > wss_sample:
        rng_wss = np.random.default_rng(1)
        cluster_sample = rng_wss.choice(nonempty_ids, size=wss_sample, replace=False)
    else:
        cluster_sample = nonempty_ids

    vectors_mmap = np.memmap(
        str(root / "vectors.dat"), dtype=np.float32, mode="r", shape=(n, dim)
    )

    total_wss   = 0.0
    total_vecs_sampled = 0

    for cl_id in cluster_sample:
        vec_ids  = load_posting(postings_dir, int(cl_id)).astype(np.int64)
        if vec_ids.size == 0:
            continue
        centroid = level2[cl_id]   # already normalised
        vecs     = vectors_mmap[vec_ids]   # (k, dim)
        cos_sims = vecs @ centroid         # (k,)
        total_wss += float((1.0 - cos_sims).sum())
        total_vecs_sampled += len(vec_ids)

    avg_wss_per_vec = total_wss / total_vecs_sampled if total_vecs_sampled > 0 else float("nan")

    # ── Silhouette (sampled) ──────────────────────────────────────────────────
    # For a random sample of vectors, compute silhouette using L2 cluster assignments.
    # a(i) = avg cosine distance to other vectors in same cluster
    # b(i) = avg cosine distance to all vectors in nearest other cluster
    # s(i) = (b-a) / max(a,b)
    # We approximate b(i) using centroid distances for speed.
    sil_scores = []
    if n >= SILHOUETTE_SAMPLES and len(nonempty_ids) >= 2:
        rng_sil = np.random.default_rng(2)
        # Pick which clusters to pull samples from (proportional to size)
        sil_cluster_ids = rng_sil.choice(
            nonempty_ids,
            size=min(100, len(nonempty_ids)),
            replace=False,
            p=l2_sizes[nonempty_ids] / l2_sizes[nonempty_ids].sum(),
        )
        for cl_id in sil_cluster_ids:
            vec_ids = load_posting(postings_dir, int(cl_id)).astype(np.int64)
            if len(vec_ids) < 2:
                continue
            sample_n = min(20, len(vec_ids))
            sampled  = rng_sil.choice(vec_ids, size=sample_n, replace=False)
            vecs = vectors_mmap[sampled]   # (sample_n, dim)

            # a(i): mean intra-cluster cosine distance (1 - sim)
            intra_sim = vecs @ vecs.T   # (sample_n, sample_n)
            np.fill_diagonal(intra_sim, 1.0)   # exclude self
            a = (1.0 - intra_sim).sum(axis=1) / max(sample_n - 1, 1)

            # b(i): nearest-other-cluster via centroid proxy
            all_sims    = vecs @ level2.T   # (sample_n, n_l2)
            all_sims[:, cl_id] = -2.0       # mask own cluster
            nearest_cl  = int(np.argmax(all_sims.mean(axis=0)))
            near_ids    = load_posting(postings_dir, nearest_cl).astype(np.int64)
            if near_ids.size == 0:
                continue
            near_vecs   = vectors_mmap[rng_sil.choice(near_ids, size=min(20, len(near_ids)), replace=False)]
            b = (1.0 - vecs @ near_vecs.T).mean(axis=1)

            s = (b - a) / np.maximum(a, b)
            sil_scores.extend(s.tolist())

    silhouette = float(np.mean(sil_scores)) if sil_scores else float("nan")

    del vectors_mmap

    return {
        "scale":              scale,
        "n_vectors":          n,
        "n_l1":               level1.shape[0],
        "n_l2":               n_l2,
        "empty_l2":           empty_l2,
        "size_mean":          size_mean,
        "size_std":           size_std,
        "size_cv":            size_cv,
        "size_max":           size_max,
        "size_min":           size_min,
        "inter_centroid_cos_dist": inter_cos_dist,
        "wss_per_vec":        avg_wss_per_vec,
        "silhouette":         silhouette,
        "_l2_sizes":          l2_sizes,
    }


cluster_results = {}
for scale in TARGETS:
    if not (INDEX_ROOTS[scale] / "manifest.json").exists():
        print(f"[{scale}] skip — not built")
        continue
    print(f"Computing clustering metrics for {scale}...")
    r = clustering_metrics(scale)
    cluster_results[scale] = r
    print(f"  L1={r['n_l1']}  L2={r['n_l2']}  empty_L2={r['empty_l2']}")
    print(f"  Cluster size  mean={r['size_mean']:.0f}  std={r['size_std']:.0f}  cv={r['size_cv']:.3f}  [{r['size_min']}–{r['size_max']}]")
    print(f"  Inter-centroid cosine dist (L1) = {r['inter_centroid_cos_dist']:.4f}")
    print(f"  WSS per vector (cosine dist)    = {r['wss_per_vec']:.4f}")
    print(f"  Silhouette score (sampled)      = {r['silhouette']:.4f}")
    print()

In [ ]:
# Cluster size distribution histograms
if cluster_results:
    fig, axes = plt.subplots(1, len(cluster_results), figsize=(7 * len(cluster_results), 4))
    if len(cluster_results) == 1:
        axes = [axes]

    for ax, (scale, r) in zip(axes, cluster_results.items()):
        sizes = r["_l2_sizes"]
        nonempty = sizes[sizes > 0]
        ax.hist(nonempty, bins=50, color="steelblue", edgecolor="white", linewidth=0.3)
        ax.axvline(r["size_mean"], color="red",    linestyle="--", label=f"mean={r['size_mean']:.0f}")
        ax.axvline(np.median(nonempty), color="orange", linestyle="--", label=f"median={np.median(nonempty):.0f}")
        ax.set_xlabel("Vectors per L2 cluster")
        ax.set_ylabel("Frequency")
        ax.set_title(f"L2 cluster size distribution — {scale}  (CV={r['size_cv']:.3f})")
        ax.legend()

    plt.tight_layout()
    plt.savefig(DATA_ROOT / "cluster_size_dist.png", dpi=120)
    plt.show()

In [ ]:
# L2 clusters per L1 cluster
for scale in cluster_results:
    root   = INDEX_ROOTS[scale]
    layout = json.loads((root / "layout.json").read_text())
    l2_per_l1 = [len(v) for v in layout["level1_to_level2"]]

    fig, ax = plt.subplots(figsize=(8, 3))
    ax.bar(range(len(l2_per_l1)), sorted(l2_per_l1, reverse=True), color="teal", width=1.0)
    ax.set_xlabel("L1 cluster (sorted by L2 count)")
    ax.set_ylabel("Number of L2 sub-clusters")
    ax.set_title(f"L2 sub-clusters per L1 cluster — {scale}")
    plt.tight_layout()
    plt.savefig(DATA_ROOT / f"l2_per_l1_{scale}.png", dpi=120)
    plt.show()
    print(f"[{scale}] L2-per-L1:  mean={np.mean(l2_per_l1):.1f}  max={max(l2_per_l1)}  min={min(l2_per_l1)}")

---
## Search Quality Metrics

Compares IVF results against brute-force (exact) nearest neighbours on a random query sample.

| Metric | Definition |
|--------|------------|
| **Recall@k** | `|IVF_top_k ∩ BF_top_k| / k` |
| **Mean reciprocal rank** | `1 / rank_of_first_true_positive` |
| **IVF latency (ms)** | Wall time per query |
| **BF latency (ms)** | Wall time per query (mmap full scan) |

In [ ]:
def brute_force_topk(vectors_mmap: np.ndarray, query: np.ndarray, k: int) -> list[int]:
    sims = vectors_mmap @ query
    top  = np.argpartition(sims, -k)[-k:]
    return top[np.argsort(sims[top])[::-1]].tolist()


def search_quality(index: TwoLevelIVFIndex, scale: str, top_k: int = TOP_K, n_queries: int = N_QUERY_VECTORS) -> dict:
    root    = INDEX_ROOTS[scale]
    n       = index.vector_count
    dim     = index.dimension

    queries = make_query_vectors(n=n_queries, dim=dim)   # fresh, never inserted

    vectors_mmap = np.memmap(str(root / "vectors.dat"), dtype=np.float32, mode="r", shape=(n, dim))

    recalls  = []
    mrrs     = []
    ivf_times = []
    bf_times  = []

    for q in tqdm(queries, desc=f"Evaluating {scale}", leave=False):
        # IVF query
        t0 = time.perf_counter()
        ivf_hits = [vid for vid, _ in index.query(q, top_k=top_k)]
        ivf_times.append((time.perf_counter() - t0) * 1000)

        # Brute-force
        t0 = time.perf_counter()
        bf_hits = brute_force_topk(vectors_mmap, q, top_k)
        bf_times.append((time.perf_counter() - t0) * 1000)

        # Recall@k
        ivf_set = set(ivf_hits)
        bf_set  = set(bf_hits)
        recalls.append(len(ivf_set & bf_set) / top_k)

        # MRR: rank of the best BF result inside IVF list
        mrr_val = 0.0
        for rank, vid in enumerate(ivf_hits, start=1):
            if vid in bf_set:
                mrr_val = 1.0 / rank
                break
        mrrs.append(mrr_val)

    del vectors_mmap

    return {
        "scale":           scale,
        "recall_at_k":     float(np.mean(recalls)),
        "mrr":             float(np.mean(mrrs)),
        "ivf_latency_ms":  float(np.mean(ivf_times)),
        "ivf_p99_ms":      float(np.percentile(ivf_times, 99)),
        "bf_latency_ms":   float(np.mean(bf_times)),
        "_recalls":        recalls,
        "_ivf_times":      ivf_times,
    }


search_results = {}
# Check 1M
if (INDEX_ROOTS["1M"] / "manifest.json").exists():
    idx_1M_eval = TwoLevelIVFIndex(root_dir=str(INDEX_ROOTS["1M"]), **IVF_KWARGS)
    if idx_1M_eval.vector_count > 0:
        print(f"Running search eval on 1M index ({idx_1M_eval.vector_count:,} vectors)...")
        r = search_quality(idx_1M_eval, "1M")
        search_results["1M"] = r
        print(f"  Recall@{TOP_K}  = {r['recall_at_k']:.4f}")
        print(f"  MRR       = {r['mrr']:.4f}")
        print(f"  IVF latency (mean/p99) = {r['ivf_latency_ms']:.2f} ms / {r['ivf_p99_ms']:.2f} ms")
        print(f"  BF  latency (mean)     = {r['bf_latency_ms']:.2f} ms")
        print()

# Check 10M
if (INDEX_ROOTS["10M"] / "manifest.json").exists():
    idx_10M_eval = TwoLevelIVFIndex(root_dir=str(INDEX_ROOTS["10M"]), **IVF_KWARGS)
    if idx_10M_eval.vector_count > 0:
        print(f"Running search eval on 10M index ({idx_10M_eval.vector_count:,} vectors)...")
        r = search_quality(idx_10M_eval, "10M")
        search_results["10M"] = r
        print(f"  Recall@{TOP_K}  = {r['recall_at_k']:.4f}")
        print(f"  MRR       = {r['mrr']:.4f}")
        print(f"  IVF latency (mean/p99) = {r['ivf_latency_ms']:.2f} ms / {r['ivf_p99_ms']:.2f} ms")
        print(f"  BF  latency (mean)     = {r['bf_latency_ms']:.2f} ms")

In [ ]:
# Recall distribution + latency CDF
if search_results:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    colors = {"1M": "steelblue", "10M": "darkorange"}

    # Recall histogram
    ax = axes[0]
    for scale, r in search_results.items():
        ax.hist(r["_recalls"], bins=20, alpha=0.6, label=f"{scale} (mean={r['recall_at_k']:.3f})",
                color=colors[scale], edgecolor="white")
    ax.set_xlabel(f"Recall@{TOP_K}")
    ax.set_ylabel("Query count")
    ax.set_title(f"Recall@{TOP_K} distribution")
    ax.legend()

    # IVF latency CDF
    ax = axes[1]
    for scale, r in search_results.items():
        times = sorted(r["_ivf_times"])
        cdf   = np.arange(1, len(times) + 1) / len(times)
        ax.plot(times, cdf, label=f"{scale}", color=colors[scale])
    ax.set_xlabel("IVF query latency (ms)")
    ax.set_ylabel("CDF")
    ax.set_title("IVF query latency CDF")
    ax.legend()

    plt.tight_layout()
    plt.savefig(DATA_ROOT / "search_quality.png", dpi=120)
    plt.show()

---
## Summary

In [ ]:
print(f"{'Metric':<38} {'1M':>12} {'10M':>12}")
print("─" * 64)

def row(label, key_fn, fmt="{}", missing="—"):
    vals = []
    for scale in ("1M", "10M"):
        try:
            vals.append(fmt.format(key_fn(scale)))
        except Exception:
            vals.append(missing)
    print(f"{label:<38} {vals[0]:>12} {vals[1]:>12}")

row("Vectors inserted",
    lambda s: f"{cluster_results[s]['n_vectors']:,}", "{}")
row("L1 clusters",
    lambda s: cluster_results[s]["n_l1"])
row("L2 clusters",
    lambda s: cluster_results[s]["n_l2"])
row("Empty L2 clusters",
    lambda s: cluster_results[s]["empty_l2"])
row("Cluster size mean",
    lambda s: f"{cluster_results[s]['size_mean']:.0f}", "{}")
row("Cluster size CV (lower=better)",
    lambda s: f"{cluster_results[s]['size_cv']:.3f}", "{}")
row("Inter-L1-centroid dist (higher=better)",
    lambda s: f"{cluster_results[s]['inter_centroid_cos_dist']:.4f}", "{}")
row("WSS per vector (lower=better)",
    lambda s: f"{cluster_results[s]['wss_per_vec']:.4f}", "{}")
row("Silhouette score (higher=better)",
    lambda s: f"{cluster_results[s]['silhouette']:.4f}", "{}")
row(f"Recall@{TOP_K} (higher=better)",
    lambda s: f"{search_results[s]['recall_at_k']:.4f}", "{}")
row("MRR (higher=better)",
    lambda s: f"{search_results[s]['mrr']:.4f}", "{}")
row("IVF latency mean (ms)",
    lambda s: f"{search_results[s]['ivf_latency_ms']:.2f}", "{}")
row("IVF latency p99 (ms)",
    lambda s: f"{search_results[s]['ivf_p99_ms']:.2f}", "{}")
row("BF latency mean (ms)",
    lambda s: f"{search_results[s]['bf_latency_ms']:.2f}", "{}")
row("Total storage",
    lambda s: fmt_bytes(reports[s]["sizes_bytes"]["TOTAL"]), "{}")